# 1. GOTOBI USDJPY - HYPOTHESIS DEFINITION & BASELINE BACKTEST

This notebook formulates the quantitative hypothesis and establishes the baseline backtest for the **Gotobi effect** on **5-minute Axi USDJPY CFD bars (UTC, converted from the MT5 server time)**.

### 1.1 CORE QUANTITATIVE HYPOTHESIS
- **Payment-day flow**: on Japanese *gotobi* days (the 5th, 10th, 15th, 20th, 25th and the last day of the month) Japanese importers settle in USD before the **09:55 JST Tokyo fixing**. The structural USD demand should lift USDJPY between the Tokyo open and the fixing. Tested by window return in section 4.1.
- **Only on gotobi days**: the same window on the neighbouring business days (placebos at -2, -1, +1, +2) should show no comparable drift. If it does, the edge is a Tokyo-morning effect, not a gotobi effect. Tested in section 4.2.
- **Not one day driving the result**: the effect should be spread across the six payment dates, not carried by a single one. Tested in section 4.3.
- **Shifted days are still gotobi days**: when a payment day falls on a weekend or holiday, settlement moves to the previous business day, so the shifted days should carry the effect too. Tested in section 4.4.

### 1.2 FROZEN STRATEGY RULES
| Rule Component | Specification |
| :--- | :--- |
| **Instrument** | USDJPY CFD (Axi), 5-minute bars (UTC), Axi MT5 history |
| **Calendar Filter** | JST day-of-month in {5, 10, 15, 20, 25} or last calendar day of the month |
| **Shift** | If the gotobi day is not a business day, trade the **previous** business day |
| **Entry** | Buy at the **OPEN** of the 07:30 JST bar (22:30 UTC of the previous calendar day) |
| **Exit** | Sell at the **OPEN** of the 09:55 JST bar (00:55 UTC), the Tokyo fixing |
| **Direction** | Long only, **no stop, no take profit** (position size is the risk management) |
| **Trading Costs** | Feed spread x `spread_scale` per trade (placeholder until the real Axi spread at 22:30 UTC is measured); robustness with a flat 0.7 and 1.0 pips |
| **Position Sizing** | (a) **100 % of capital** (per-trade reference) - (b) **`size_fraction` of capital** (what the robot trades) |

### 1.3 PROVENANCE & VALIDATION DESIGN
- **Naked window**: the same 07:30-09:55 JST window on every business day, no calendar filter. The control group for everything below.
- **Placebo windows**: the gotobi rule moved by -2, -1, +1 and +2 business days.
- **Multiple testing**: 16 hypotheses were tested while building this variant, so the acceptance bar for the net t-stat is raised to `min_t_net` (2.5) instead of the usual 2.0.
- **Split**: IS 2010-2022, OOS from 2023-01-01. Indicators are not used, so nothing is fitted in-sample: IS and OOS differ only in the period.

> **HOLIDAY NOTE**: only weekends are modelled as non-business days here (no Japanese holiday calendar). On a Japanese holiday the market still trades, the payment day just moves. Section 6 compares the dates the engine trades with the dates rebuilt in this notebook.

### 1.4 STRATEGY PARAMETERS

In [1]:
VARIANT = "v1_axi_usdjpy"

## 2. IMPORTS AND CONFIG

In [2]:
import os
import sys
import json
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ---- Paths ----
def find_upwards(name, start=None, must_be_dir=True):
    """Walk up from start (cwd by default) and return the first folder that contains `name`."""
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        target = base / name
        if target.is_dir() if must_be_dir else target.exists():
            return base
    raise FileNotFoundError(f"Could not find '{name}' walking up from {here}")

EDGE_DIR   = find_upwards('variants')                  # edge folder (notebooks, engine and variants/ live together)
utils_path = str(find_upwards('utils') / 'utils')      # research_development/utils
for p in (utils_path, str(EDGE_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

from utils import compute_metrics
from metrics import sharpe_ratio, max_drawdown
import gotobi_engine as eng
import variant_config
from variant_config import load_variant, save_json

importlib.reload(eng)   # resets ENGINE_DEFAULTS: nothing leaks between variants

variant_config.EDGE_DIR = EDGE_DIR
assert (variant_config.EDGE_DIR / "variants").is_dir(), f"Doesn't exist {variant_config.EDGE_DIR / 'variants'}"

# ---- Variant config: single source of truth ----
cfg        = load_variant(VARIANT)
eng.set_engine_defaults(**cfg.get("engine_defaults", {}))
RUN_DIR    = cfg["run_dir"]
TICKER     = cfg["instrument"]["ticker"]
DATA_START = cfg["instrument"]["start"]
OOS_START  = cfg["oos_start_ts"]
P          = cfg["params"]
GOTOBI_DAYS, SHIFT, ENTRY_JST, EXIT_JST = P["gotobi_days"], P["shift"], P["entry_jst"], P["exit_jst"]
PLACEBOS   = P["placebo_offsets"]
COST_MODE, SPREAD_SCALE = cfg["costs"]["cost_mode"], cfg["costs"]["spread_scale"]
FLAT_PIPS  = cfg["costs"]["robustness_flat_pips"]
EV         = cfg["evidence"]
HARD_LIMIT, CAPITAL = EV["hard_limit"], float(EV["capital"])
SIZE_FRAC  = eng.ENGINE_DEFAULTS["size_fraction"]
CONTRACT_SIZE, MIN_LOT, LOT_STEP = (eng.ENGINE_DEFAULTS[k] for k in ("contract_size", "min_lot", "lot_step"))
PPY        = 252    # FX trades on business days: annualisation for Sharpe
PIP        = 0.01   # USDJPY pip size

# Frozen rules, passed as-is to every backtest call in this notebook
RULES = dict(gotobi_days=GOTOBI_DAYS, shift=SHIFT, entry_jst=ENTRY_JST, exit_jst=EXIT_JST,
             cost_mode=COST_MODE, spread_scale=SPREAD_SCALE)

# Return-series helpers: the engine shares the API of monday_effect_engine; the per-bar series may be named differently
bar_returns = getattr(eng, "bar_returns", getattr(eng, "hourly_returns", None))
assert bar_returns is not None, "gotobi_engine needs bar_returns() or hourly_returns()"

# ---- Acceptance thresholds: every PASS / FAIL in this notebook is judged against these ----
PF_MIN, T_MIN, N_MIN = 1.3, EV["min_t_net"], 300   # hypothesis: Profit Factor, net t-stat of the mean trade (raised for 16 hypotheses), trades
N_OOS_MIN            = 60                          # minimum OOS trades to say anything
ALPHA                = 0.05                        # significance level of the statistical tests
T_PLACEBO_MAX        = 2.0                         # a placebo window with t above this means the effect is not specific to gotobi
MAX_SIGNAL_LOSS      = 0.05                        # max share of expected gotobi dates without a tradable window (gaps)
MIN_TRADES_PER_YEAR  = 40                          # every full calendar year should have at least this many trades (about 72 expected)
YEARS_POS_MIN        = 0.75                        # share of full calendar years that must be profitable

# ---- PASS / FAIL registry: each section registers its checks here, section 8 groups them ----
CHECKS = []

def check(section, name, passed, detail="", quiet=False):
    """Register one check. passed: True -> PASS, False -> FAIL, None -> SKIP (not applicable).
    Re-running a cell overwrites its previous entries instead of duplicating them."""
    status = "SKIP" if passed is None else ("PASS" if bool(passed) else "FAIL")
    entry = {"section": section, "name": name, "status": status, "detail": detail}
    for i, c in enumerate(CHECKS):
        if c["section"] == section and c["name"] == name:
            CHECKS[i] = entry
            break
    else:
        CHECKS.append(entry)
    if not quiet:
        print(f"   [{status}] {name}" + (f"  ->  {detail}" if detail else ""))
    return status

print("Imports complete.")
print(f"Loaded gotobi_engine from: {eng.__file__}")
print(f"Edge dir: {EDGE_DIR}")
print(f"Variant: {cfg['id']} | {TICKER} M5 | gotobi days {GOTOBI_DAYS} | shift={SHIFT} | {ENTRY_JST}-{EXIT_JST} JST | size {SIZE_FRAC:.0%}")
print(f"Costs: {COST_MODE} x {SPREAD_SCALE:g} | robustness flat pips {FLAT_PIPS} | hypotheses tested {EV['hypotheses_tested']} -> min net t = {T_MIN:g}")
print(f"CFD: contract {CONTRACT_SIZE:g} | min lot {MIN_LOT:g} | step {LOT_STEP:g} | capital {CAPITAL:,.0f} | hard limit -{HARD_LIMIT:.0%}")
print(f"Run folder: {RUN_DIR}")

ModuleNotFoundError: No module named 'gotobi_engine'

## 3. LOAD DATA AND IN-SAMPLE / OUT-OF-SAMPLE

In [ ]:
df_raw = eng.load_data(TICKER, start=DATA_START)
print(len(df_raw), "M5 bars |", df_raw.index[0], "->", df_raw.index[-1])
print(df_raw.groupby(df_raw.index.year).size().to_dict())
_step_m = df_raw.index.to_series().diff().dt.total_seconds().div(60).dropna()
print(f"Steps different from 5 min: {int((_step_m != 5).sum())} | largest gap: {_step_m.max() / 60:.1f} h")

# The engine adds whatever it needs (window open/close, spread, gotobi flag) in prepare()
df = eng.prepare(df_raw, **RULES)

n_is = int((df.index < OOS_START).sum())
df_is, df_oos = df.iloc[:n_is].copy(), df.iloc[n_is:].copy()
assert len(df_is) > 0 and len(df_oos) > 0, "Check split.oos_start against the history of this ticker."
oos_start = df_oos.index[0]

# ---- Gotobi calendar rebuilt in this notebook (JST dates), independent of the engine ----
def gotobi_dates(start, end, days=GOTOBI_DAYS, shift=SHIFT):
    """JST trade dates: day-of-month in `days` ('eom' = last calendar day), moved to the previous business day if on a weekend."""
    out = []
    for m in pd.period_range(start, end, freq="M"):
        for d in days:
            day = m.end_time.normalize() if d == "eom" else pd.Timestamp(year=m.year, month=m.month, day=int(d))
            if shift == "previous":
                while day.dayofweek >= 5:
                    day -= pd.Timedelta(days=1)
            elif day.dayofweek >= 5:
                continue
            out.append(day)
    return pd.DatetimeIndex(sorted(set(out)))

def window_times(jst_date):
    """UTC timestamps of the entry and exit bars for a JST trade date (JST = UTC+9, no DST)."""
    t_in  = pd.Timestamp(f"{jst_date:%Y-%m-%d} {ENTRY_JST}") - pd.Timedelta(hours=9)
    t_out = pd.Timestamp(f"{jst_date:%Y-%m-%d} {EXIT_JST}")  - pd.Timedelta(hours=9)
    return t_in, t_out

def tstat(x):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    return float(x.mean() / (x.std(ddof=1) / np.sqrt(len(x)))) if len(x) > 2 and x.std(ddof=1) > 0 else np.nan

def window_returns(jst_dates):
    """Gross open-to-open return (%) of the 07:30-09:55 JST window for each JST date that has both bars."""
    o = df_raw["open"]
    rows = []
    for d in jst_dates:
        t_in, t_out = window_times(d)
        if t_in in o.index and t_out in o.index:
            rows.append((d, (o.at[t_out] / o.at[t_in] - 1) * 100, o.at[t_in]))
    return pd.DataFrame(rows, columns=["jst_date", "ret", "price_in"]).set_index("jst_date")

START_D = max(pd.Timestamp(DATA_START), df_raw.index[0].normalize() + pd.Timedelta(days=2))
END_D   = df_raw.index[-1].normalize() - pd.Timedelta(days=1)
G_DATES = gotobi_dates(START_D, END_D)
G_DATES = G_DATES[(G_DATES >= START_D) & (G_DATES <= END_D)]
g_win = window_returns(G_DATES)

print("=" * 70)
print("DATA INGESTION & DATASET SPLIT")
print("=" * 70)
print(f"Total Prepared Bars: {len(df):,} ({df.index.min():%Y-%m-%d} to {df.index.max():%Y-%m-%d})")
print(f"In-Sample (IS, {len(df_is)/len(df)*100:.1f}%): {len(df_is):,} bars ({df_is.index.min():%Y-%m-%d} to {df_is.index.max():%Y-%m-%d})")
print(f"Out-of-Sample (OOS, {len(df_oos)/len(df)*100:.1f}%): {len(df_oos):,} bars ({df_oos.index.min():%Y-%m-%d} to {df_oos.index.max():%Y-%m-%d})")
print(f"Gotobi dates expected: {len(G_DATES)} | with both window bars present: {len(g_win)}")
print("=" * 70)

print("CHECKS (section 3):")
lost = len(G_DATES) - len(g_win)
check("3. Data", "IS and OOS are both non-empty", len(df_is) > 0 and len(df_oos) > 0, f"IS {len(df_is):,} bars | OOS {len(df_oos):,} bars")
check("3. Data", f"Gotobi dates without a tradable window <= {MAX_SIGNAL_LOSS:.0%}", lost / len(G_DATES) <= MAX_SIGNAL_LOSS,
      f"{lost} of {len(G_DATES)} dates lost to gaps ({lost / len(G_DATES):.1%})")
if lost:
    miss = [d for d in G_DATES if d not in g_win.index]
    print("   Dates without bars (first 15):", [f"{d:%Y-%m-%d}" for d in miss[:15]])
df[["open", "high", "low", "close"]].tail()

## 4. EXPLORATORY CHECKS OF THE HYPOTHESIS (EDGE-SPECIFIC)

The four claims of section 1.1, checked on the data before any backtest. **Gross** = no costs, **net** = after a flat `FLAT_PIPS[0]` pips round trip (the engine's feed-based costs are used from section 5 on).

### 4.1 IS THE GOTOBI WINDOW SPECIAL? GOTOBI VS EVERY OTHER BUSINESS DAY

In [ ]:
SUBPERIODS = {"2010-2014": (2010, 2014), "2015-2019": (2015, 2019), "2020-2022": (2020, 2022), "OOS 2023+": (2023, 2100)}
def period_of(y):
    return next(k for k, (a, b) in SUBPERIODS.items() if a <= y <= b)

all_bd = pd.bdate_range(START_D, END_D)
a_win = window_returns(all_bd)
a_win["gotobi"] = a_win.index.isin(G_DATES)
a_win["period"] = [period_of(y) for y in a_win.index.year]
a_win["net"] = a_win["ret"] - FLAT_PIPS[0] * PIP / a_win["price_in"] * 100
g_all, o_all = a_win[a_win["gotobi"]], a_win[~a_win["gotobi"]]

rows = []
for per in list(SUBPERIODS) + ["All"]:
    g = g_all if per == "All" else g_all[g_all["period"] == per]
    o = o_all if per == "All" else o_all[o_all["period"] == per]
    rows.append({"Period": per, "Gotobi N": len(g), "Gotobi gross %": g["ret"].mean(), "Gotobi t": tstat(g["ret"]),
                 "Other N": len(o), "Other gross %": o["ret"].mean(), "Other t": tstat(o["ret"])})
win_tbl = pd.DataFrame(rows).set_index("Period")
print("=" * 70)
print("07:30-09:55 JST WINDOW RETURN: GOTOBI DAYS VS OTHER BUSINESS DAYS (% gross)")
print("=" * 70)
display(win_tbl.round(4))

print("\nCHECKS (4.1):")
t_gw, p_gw = stats.ttest_ind(g_all["ret"], o_all["ret"], equal_var=False, alternative="greater")
check("4.1 Window", "Gotobi window mean return > 0 in every sub-period", (win_tbl.loc[list(SUBPERIODS), "Gotobi gross %"] > 0).all(),
      "; ".join(f"{p}: {win_tbl.loc[p, 'Gotobi gross %']:+.4f}%" for p in SUBPERIODS))
check("4.1 Window", f"Gotobi beats the other business days (one-sided Welch p < {ALPHA:g})", p_gw < ALPHA,
      f"gotobi {g_all['ret'].mean():+.4f}% vs other {o_all['ret'].mean():+.4f}% | t = {t_gw:.2f}, p = {p_gw:.4f}")
check("4.1 Window", f"Net window drift is significant (t >= {T_MIN:g})", tstat(g_all["net"]) >= T_MIN,
      f"net mean {g_all['net'].mean():+.4f}% | t = {tstat(g_all['net']):.2f} (flat {FLAT_PIPS[0]:g} pips)")
check("4.1 Window", "Gotobi drift is not present in the other business days (t < 2)", tstat(o_all["ret"]) < 2.0,
      f"other-days t = {tstat(o_all['ret']):.2f}")

### 4.2 PLACEBO WINDOWS: THE SAME RULE MOVED BY -2, -1, +1, +2 BUSINESS DAYS

In [ ]:
from pandas.tseries.offsets import BDay
plc_rows, plc_series = [], {}
for k in [0] + list(PLACEBOS):
    d_k = pd.DatetimeIndex([d + BDay(k) for d in G_DATES])
    d_k = d_k[(d_k >= START_D) & (d_k <= END_D)]
    w = window_returns(d_k)
    plc_series[k] = w["ret"]
    plc_rows.append({"Offset (bdays)": k, "N": len(w), "Gross mean %": w["ret"].mean(), "Gross t": tstat(w["ret"]),
                     "Hit rate %": (w["ret"] > 0).mean() * 100})
plc_tbl = pd.DataFrame(plc_rows).set_index("Offset (bdays)")
print("=" * 70)
print("PLACEBO WINDOWS (offset 0 = the real gotobi window)")
print("=" * 70)
display(plc_tbl.round(4))

print("\nCHECKS (4.2):")
others = plc_tbl.drop(index=0)
check("4.2 Placebos", "Real window has the highest mean of all offsets", plc_tbl["Gross mean %"].idxmax() == 0,
      f"best offset = {plc_tbl['Gross mean %'].idxmax()} | real {plc_tbl.loc[0, 'Gross mean %']:+.4f}% vs best placebo {others['Gross mean %'].max():+.4f}%")
check("4.2 Placebos", f"No placebo window is significant (t < {T_PLACEBO_MAX:g})", (others["Gross t"] < T_PLACEBO_MAX).all(),
      "; ".join(f"{k:+d}: t = {v:.2f}" for k, v in others["Gross t"].items()))
check("4.2 Placebos", "Real window mean is at least 2x the best placebo", plc_tbl.loc[0, "Gross mean %"] >= 2 * others["Gross mean %"].max(),
      f"{plc_tbl.loc[0, 'Gross mean %']:+.4f}% vs {others['Gross mean %'].max():+.4f}%")

fig, ax = plt.subplots(figsize=(8, 3.8), facecolor="#15171c")
ax.set_facecolor("#1b1e26")
ax.bar([str(k) for k in plc_tbl.index], plc_tbl["Gross mean %"], color=["#34d399" if k == 0 else "#9aa0ab" for k in plc_tbl.index], width=0.55)
ax.axhline(0, color="#9aa0ab", lw=0.6)
ax.set_title("Window mean return by offset in business days (%, gross)", color="#e8e9ec", fontweight="bold", loc="left")
ax.tick_params(colors="#9aa0ab")
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

### 4.3 WHICH PAYMENT DATE CARRIES THE EDGE? (BY DAY OF THE MONTH)

In [ ]:
g_all2 = g_all.copy()
def bucket(d):
    if d == (d + pd.offsets.MonthEnd(0)):
        return "eom"
    return str(d.day)
# label by the *nominal* gotobi day: a shifted date belongs to the payment day it was moved from
def nominal_label(d):
    for k in range(0, 4):
        c = d + pd.Timedelta(days=k)
        if c.day in (5, 10, 15, 20, 25):
            return str(c.day)
        if c == c + pd.offsets.MonthEnd(0):
            return "eom"
    return str(d.day)
g_all2["bucket"] = [nominal_label(d) for d in g_all2.index]
order = ["5", "10", "15", "20", "25", "eom"]
dom = g_all2.groupby("bucket")["net"].agg(["size", "mean"]).reindex(order)
dom["t"] = g_all2.groupby("bucket")["net"].apply(tstat).reindex(order)
dom["hit %"] = g_all2.groupby("bucket")["net"].apply(lambda s: (s > 0).mean() * 100).reindex(order)
dom.columns = ["N", "Net mean %", "Net t", "Hit rate %"]
print("=" * 70)
print(f"NET WINDOW RETURN BY PAYMENT DATE (flat {FLAT_PIPS[0]:g} pips)")
print("=" * 70)
display(dom.round(4))

print("\nCHECKS (4.3):")
check("4.3 Payment dates", "Net mean is positive on every payment date", (dom["Net mean %"] > 0).all(),
      "; ".join(f"{i}: {v:+.4f}%" for i, v in dom["Net mean %"].items()))
share = dom["Net mean %"].clip(lower=0) * dom["N"]
top_share = share.max() / share.sum() if share.sum() > 0 else np.nan
check("4.3 Payment dates", "No single payment date carries more than 40% of the net edge", top_share <= 0.40,
      f"top date = {share.idxmax()} with {top_share:.0%} of the net edge")
check("4.3 Payment dates", "At least 4 of the 6 payment dates are individually positive with t > 1", int(((dom['Net mean %'] > 0) & (dom['Net t'] > 1)).sum()) >= 4,
      f"{int(((dom['Net mean %'] > 0) & (dom['Net t'] > 1)).sum())} of 6")

fig, ax = plt.subplots(figsize=(8, 3.8), facecolor="#15171c")
ax.set_facecolor("#1b1e26")
ax.bar(dom.index, dom["Net mean %"], color=["#34d399" if v > 0 else "#f0555a" for v in dom["Net mean %"]], width=0.55)
ax.axhline(0, color="#9aa0ab", lw=0.6)
ax.set_title("Net window return by payment date (%)", color="#e8e9ec", fontweight="bold", loc="left")
ax.tick_params(colors="#9aa0ab")
for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.show()

### 4.4 SHIFTED PAYMENT DAYS VS DAYS THAT FELL ON A BUSINESS DAY

In [ ]:
def is_shifted(d):
    return any((d + pd.Timedelta(days=k)).day in (5, 10, 15, 20, 25) and (d + pd.Timedelta(days=k)).dayofweek >= 5 for k in (1, 2)) or \
           ((d + pd.Timedelta(days=1)) == ((d + pd.Timedelta(days=1)) + pd.offsets.MonthEnd(0)) and (d + pd.Timedelta(days=1)).dayofweek >= 5) or \
           ((d + pd.Timedelta(days=2)) == ((d + pd.Timedelta(days=2)) + pd.offsets.MonthEnd(0)) and (d + pd.Timedelta(days=2)).dayofweek >= 5)
g_all2["shifted"] = [is_shifted(d) for d in g_all2.index]
sh = g_all2.groupby("shifted")["net"].agg(["size", "mean"])
sh["t"] = g_all2.groupby("shifted")["net"].apply(tstat)
sh.index = ["On the date" if not i else "Shifted (weekend)" for i in sh.index]
sh.columns = ["N", "Net mean %", "Net t"]
print("=" * 70)
print("SHIFTED VS NOT SHIFTED (net window return)")
print("=" * 70)
display(sh.round(4))

print("\nCHECKS (4.4):")
if g_all2["shifted"].sum() >= 20:
    s_ret, n_ret = g_all2.loc[g_all2["shifted"], "net"], g_all2.loc[~g_all2["shifted"], "net"]
    check("4.4 Shift", "Shifted payment days also have a positive net mean", s_ret.mean() > 0, f"shifted {s_ret.mean():+.4f}% (N = {len(s_ret)})")
    check("4.4 Shift", "Shifted days are not significantly worse than the others (one-sided Welch p > 0.05)",
          stats.ttest_ind(s_ret, n_ret, equal_var=False, alternative="less")[1] > ALPHA,
          f"shifted {s_ret.mean():+.4f}% vs on-date {n_ret.mean():+.4f}%")
else:
    check("4.4 Shift", "Shifted payment days behave like the rest", None, f"only {int(g_all2['shifted'].sum())} shifted dates")

## 5. BASELINE BACKTEST WITH FROZEN RULES

In [ ]:
trades_is  = eng.run_gotobi_backtest(df_is,  **RULES)
trades_oos = eng.run_gotobi_backtest(df_oos, **RULES)
trades_all = eng.run_gotobi_backtest(df,     **RULES)

def sized(t, sizing):
    return t if sizing == "capital" else t.assign(pnl=t["pnl_frac"], win=t["pnl_frac"] > 0)

def perf_row(t, price_df, period, sizing):
    m = compute_metrics(sized(t, sizing), contract_multiplier=1.0, cost_per_trade=0.0)
    ret = eng.daily_returns(price_df, t, sizing=sizing)
    eq = (1.0 + ret).cumprod()
    yrs = (price_df.index[-1] - price_df.index[0]).days / 365.25
    return {"Period": period, "Sizing": "100% Capital" if sizing == "capital" else f"{SIZE_FRAC:.0%} Capital (robot)",
            "Trades": m["num_trades"], "Win Rate": f"{m['win_rate']:.1f}%", "Profit Factor": f"{m['profit_factor']:.3f}",
            "Expectancy": f"{m['expectancy']:.4f}%", "Total Return": f"{(eq.iloc[-1] - 1) * 100:.1f}%",
            "CAGR": f"{((eq.iloc[-1] ** (1 / yrs)) - 1) * 100:.1f}%" if yrs > 0 else "n/a",
            "Sharpe": f"{sharpe_ratio(ret, periods_per_year=PPY):.2f}", "Max Drawdown": f"{max_drawdown(eq) * 100:.2f}%"}

periods = [(f"In-Sample ({df_is.index[0].year}-{df_is.index[-1].year})", trades_is, df_is),
           (f"Out-of-Sample ({df_oos.index[0].year}-{df_oos.index[-1].year})", trades_oos, df_oos),
           (f"Full Period ({df.index[0].year}-{df.index[-1].year})", trades_all, df)]
perf_table = pd.DataFrame([perf_row(t, d, name, s) for name, t, d in periods for s in ("capital", "fraction")])

print("=" * 80)
print(f"PERFORMANCE SUMMARY (NET OF COSTS: {COST_MODE} x {SPREAD_SCALE:g} · daily MTM drawdown, Sharpe annualised x{PPY})")
print("=" * 80)
display(perf_table)

BG, PANEL, GRIDC, TXT, SUB, GREEN, RED, ORANGE = "#15171c", "#1b1e26", "#2a2d37", "#e8e9ec", "#9aa0ab", "#34d399", "#f0555a", "#f5a524"
def dark_ax(ax):
    ax.set_facecolor(PANEL); ax.grid(True, color=GRIDC, lw=0.8, alpha=0.9)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.tick_params(colors=SUB, labelsize=9)

ret_cap = eng.daily_returns(df, trades_all, sizing="capital")
ret_fr = eng.daily_returns(df, trades_all, sizing="fraction")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, facecolor=BG)
for ax, r, col, lab in ((ax1, ret_cap, GREEN, "100% Capital Sizing"), (ax2, ret_fr, "#5468ff", f"{SIZE_FRAC:.0%} Capital Sizing (robot)")):
    dark_ax(ax)
    ax.plot(r.index, (1 + r).cumprod(), color=col, lw=1.6, label=lab)
    ax.axvline(oos_start, color=ORANGE, ls="--", lw=1.2, label=f"OOS Split Date ({oos_start:%Y-%m-%d})")
    ax.set_title(f"{TICKER} Gotobi - Cumulative Equity Curve ({lab})", color=TXT, fontweight="bold", loc="left")
    ax.set_ylabel("Equity Multiplier", color=SUB)
    ax.legend(frameon=False, labelcolor=SUB, fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

profit_usd = CAPITAL * ((1.0 + ret_fr).cumprod() - 1.0)
price_daily = df["close"].resample("D").last().reindex(ret_fr.index).ffill().bfill()
buy_hold = CAPITAL * (price_daily / price_daily.iloc[0] - 1.0)
entries = trades_all.assign(entry_time=pd.to_datetime(trades_all["entry_time"])).set_index("entry_time")
won = entries["pnl"] > 0

fig, (ax_px, ax_pl) = plt.subplots(2, 1, figsize=(14, 8), sharex=True, facecolor=BG, gridspec_kw={"height_ratios": [3, 2]})
for ax in (ax_px, ax_pl):
    dark_ax(ax); ax.grid(False); ax.axvline(oos_start, color=ORANGE, ls="--", lw=1.2)
ax_px.plot(price_daily.index, price_daily, color=SUB, lw=1.2, label=f"{TICKER} close (daily)")
ax_px.scatter(entries.index[won], entries.loc[won, "entry_price"], marker="^", s=10, color=GREEN, zorder=3, label="Winning trade")
ax_px.scatter(entries.index[~won], entries.loc[~won, "entry_price"], marker="v", s=10, color=RED, zorder=3, label="Losing trade")
ax_px.set_title(f"{TICKER} price vs money earned by the strategy", color=TXT, fontweight="bold", loc="left")
ax_px.set_ylabel("Price", color=SUB)
ax_px.legend(frameon=False, labelcolor=SUB, fontsize=9, loc="upper left")
ax_pl.plot(profit_usd.index, profit_usd, color=GREEN, lw=1.6, label=f"Strategy profit ({SIZE_FRAC:.0%} sizing, capital {CAPITAL:,.0f})")
ax_pl.fill_between(profit_usd.index, profit_usd, 0, color=GREEN, alpha=0.15)
ax_pl.axhline(0, color=SUB, lw=0.8)
ax_pl.set_ylabel("Money earned", color=SUB)
ax_bh = ax_pl.twinx()
ax_bh.plot(buy_hold.index, buy_hold, color=SUB, lw=1.0, ls="--", label="Buy & hold on the same capital (right axis)")
ax_bh.grid(False); ax_bh.tick_params(colors=SUB)
for spine in ax_bh.spines.values():
    spine.set_visible(False)
h1, l1 = ax_pl.get_legend_handles_labels(); h2, l2 = ax_bh.get_legend_handles_labels()
ax_pl.legend(h1 + h2, l1 + l2, frameon=False, labelcolor=SUB, fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

in_market = float(trades_all["bars"].sum()) / len(df)
print(f"{TICKER} price change: {(price_daily.iloc[-1] / price_daily.iloc[0] - 1) * 100:+,.0f}% | "
      f"strategy profit: {profit_usd.iloc[-1]:,.0f} ({profit_usd.iloc[-1] / CAPITAL * 100:+.1f}% of {CAPITAL:,.0f}) | "
      f"buy & hold profit: {buy_hold.iloc[-1]:,.0f} | time in market: {in_market * 100:.2f}%")

print("\nCHECKS (5):")
m_is, m_oos, m_all = (compute_metrics(t, 1.0, 0.0) for t in (trades_is, trades_oos, trades_all))
t_all = tstat(trades_all["pnl"])
check("5. Baseline", f"Profit Factor, full period >= {PF_MIN:g}", m_all["profit_factor"] >= PF_MIN, f"PF = {m_all['profit_factor']:.3f}")
check("5. Baseline", f"Profit Factor, IS >= {PF_MIN:g}", m_is["profit_factor"] >= PF_MIN, f"PF = {m_is['profit_factor']:.3f}")
check("5. Baseline", f"Profit Factor, OOS >= {PF_MIN:g}", m_oos["profit_factor"] >= PF_MIN, f"PF = {m_oos['profit_factor']:.3f}")
check("5. Baseline", "OOS does not degrade vs IS (OOS PF >= 50% of IS PF)", m_oos["profit_factor"] >= 0.5 * m_is["profit_factor"],
      f"OOS {m_oos['profit_factor']:.3f} vs IS {m_is['profit_factor']:.3f}")
check("5. Baseline", f"Net t-stat of the mean trade >= {T_MIN:g} (raised for {EV['hypotheses_tested']} hypotheses tested)", t_all >= T_MIN, f"t = {t_all:.2f}")
check("5. Baseline", f"Trades, full period >= {N_MIN}", m_all["num_trades"] >= N_MIN, f"{m_all['num_trades']} trades")
check("5. Baseline", f"Trades, OOS >= {N_OOS_MIN}", m_oos["num_trades"] >= N_OOS_MIN, f"{m_oos['num_trades']} trades")
check("5. Baseline", "Expectancy > 0 in IS and OOS (net of costs)", m_is["expectancy"] > 0 and m_oos["expectancy"] > 0,
      f"IS {m_is['expectancy']:+.4f}% | OOS {m_oos['expectancy']:+.4f}%")

## 5.1 LOT-ADJUSTED SIZING AND HARD LIMIT (CFD)

In [ ]:
last_price = float(df["close"].iloc[-1])
min_capital = eng.min_capital_for_target(last_price, SIZE_FRAC)
max_frac = eng.max_fraction_for_limit(df, trades_all, HARD_LIMIT)
lot_sizes = eng.cfd_trade_sizes(trades_all, CAPITAL, SIZE_FRAC)
bar_ideal = bar_returns(df, trades_all, sizing="fraction")
bar_lot = bar_returns(df, trades_all, sizing=lot_sizes["size_fraction"].to_numpy())

def drawdown_series(r):
    eq = (1.0 + r).cumprod()
    return eq / np.maximum(eq.cummax(), 1.0) - 1.0

def limit_row(label, r, n_taken):
    rep = eng.hard_limit_report(r, HARD_LIMIT)
    total = (1.0 + r).prod() - 1.0
    return {"Sizing": label, "Trades taken": n_taken, "Total Return": f"{total * 100:.1f}%",
            "Max DD (bar MTM)": f"{rep['max_dd'] * 100:.2f}%", "Headroom": f"{rep['headroom'] * 100:+.2f} pts",
            "Hard limit": "PASS" if rep["passed"] else "FAIL"}

limit_table = pd.DataFrame([
    limit_row(f"{SIZE_FRAC:.0%} ideal fraction", bar_ideal, len(trades_all)),
    limit_row(f"Lot-adjusted (capital {CAPITAL:,.0f})", bar_lot, int(lot_sizes["taken"].sum()))])
print("=" * 80)
print(f"HARD LIMIT -{HARD_LIMIT:.0%} · LOT-ADJUSTED SIZING (min lot {MIN_LOT:g}, step {LOT_STEP:g})")
print("=" * 80)
display(limit_table)
print(f"Minimum capital so the minimum lot weighs <= {SIZE_FRAC:.0%} at {last_price:,.2f}: {min_capital:,.0f}")
print(f"Largest fixed fraction that keeps the bar-level drawdown inside -{HARD_LIMIT:.0%}: {max_frac:.1%} (configured: {SIZE_FRAC:.0%})")

print("\nCHECKS (5.1):")
rep_ideal, rep_lot = eng.hard_limit_report(bar_ideal, HARD_LIMIT), eng.hard_limit_report(bar_lot, HARD_LIMIT)
check("5.1 Hard limit", f"{SIZE_FRAC:.0%} ideal fraction stays inside -{HARD_LIMIT:.0%}", rep_ideal["passed"],
      f"max DD {rep_ideal['max_dd'] * 100:.2f}% | headroom {rep_ideal['headroom'] * 100:+.2f} pts")
check("5.1 Hard limit", f"Lot-adjusted sizing stays inside -{HARD_LIMIT:.0%}", rep_lot["passed"],
      f"max DD {rep_lot['max_dd'] * 100:.2f}% | headroom {rep_lot['headroom'] * 100:+.2f} pts")
check("5.1 Hard limit", "Configured fraction <= largest fraction allowed by the limit", SIZE_FRAC <= max_frac, f"{SIZE_FRAC:.0%} vs {max_frac:.1%}")
check("5.1 Hard limit", "Capital >= minimum capital for the minimum lot", CAPITAL >= min_capital, f"{CAPITAL:,.0f} vs {min_capital:,.0f}")
check("5.1 Hard limit", "Every trade is executable with the minimum lot", int(lot_sizes["taken"].sum()) == len(trades_all),
      f"{int(lot_sizes['taken'].sum())} of {len(trades_all)} trades taken")
rep_ib = eng.hard_limit_report_intrabar(df, trades_all, HARD_LIMIT, sizing=lot_sizes["size_fraction"].to_numpy())
check("5.1 Hard limit", f"Intrabar drawdown (candle lows) stays inside -{HARD_LIMIT:.0%}", rep_ib["passed"],
      f"max DD intrabar {rep_ib['max_dd'] * 100:.2f}% (closes only {rep_ib['max_dd_close'] * 100:.2f}%) | headroom {rep_ib['headroom'] * 100:+.2f} pts")

dd_ideal, dd_lot = drawdown_series(bar_ideal), drawdown_series(bar_lot)
fig, (ax_eq, ax_dd) = plt.subplots(2, 1, figsize=(14, 7), sharex=True, facecolor=BG)
for ax in (ax_eq, ax_dd):
    dark_ax(ax); ax.grid(False); ax.axvline(oos_start, color=ORANGE, ls=":", lw=1.0)
ax_eq.plot(bar_ideal.index, (1.0 + bar_ideal).cumprod(), color=SUB, lw=1.4, label=f"{SIZE_FRAC:.0%} ideal fraction")
ax_eq.plot(bar_lot.index, (1.0 + bar_lot).cumprod(), color=GREEN, lw=1.6, label=f"Lot-adjusted (capital {CAPITAL:,.0f})")
ax_eq.set_title(f"{TICKER} Gotobi - Equity, ideal vs lot-adjusted", color=TXT, fontweight="bold", loc="left")
ax_eq.set_ylabel("Equity Multiplier", color=SUB)
ax_eq.legend(frameon=False, labelcolor=SUB, fontsize=9, loc="upper left")
ax_dd.plot(dd_ideal.index, dd_ideal * 100, color=SUB, lw=1.0)
ax_dd.fill_between(dd_lot.index, dd_lot * 100, 0, color=RED, alpha=0.35)
ax_dd.axhline(-HARD_LIMIT * 100, color=ORANGE, ls="--", lw=1.2, label=f"Hard limit -{HARD_LIMIT:.0%}")
ax_dd.set_ylabel("Drawdown % (bar MTM)", color=SUB)
ax_dd.legend(frameon=False, labelcolor=SUB, fontsize=9, loc="lower left")
plt.tight_layout(); plt.show()

## 5.2 ANNUAL RETURNS AND HARD LIMIT CHECK

In [ ]:
def plot_annual_returns(annual, report, hard_limit):
    fig, (ax_ret, ax_dd) = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True, facecolor=BG)
    years = annual["year"].astype(str).to_numpy()
    bars = ax_ret.bar(years, annual["ret"] * 100, color=[GREEN if v >= 0 else RED for v in annual["ret"]])
    ax_dd.bar(years, annual["mdd"] * 100, color=[RED if b else SUB for b in annual["breach"]])
    ax_dd.axhline(-hard_limit * 100, color=ORANGE, ls="--", lw=1.2)
    for ax in (ax_ret, ax_dd):
        dark_ax(ax); ax.grid(False)
    for rect, value, partial in zip(bars, annual["ret"], annual["partial"]):
        label = f"{value:+.1%}" + ("*" if partial else "")
        ax_ret.annotate(label, (rect.get_x() + rect.get_width() / 2, value * 100), xytext=(0, 3 if value >= 0 else -3),
                        textcoords="offset points", ha="center", va="bottom" if value >= 0 else "top", color=TXT, fontsize=9)
    status = "PASS" if report["passed"] else "FAIL"
    ax_ret.set_title(f"Annual returns · hard limit -{hard_limit:.0%} · max DD {report['max_dd']:.1%} · {status}", color=TXT, fontweight="bold", loc="left")
    ax_ret.set_ylabel("Return %", color=SUB); ax_dd.set_ylabel("Max DD in year %", color=SUB)
    plt.tight_layout(); plt.show()

annual = eng.annual_returns(bar_lot, trades_all, HARD_LIMIT)
report_annual = eng.hard_limit_report(bar_lot, HARD_LIMIT)
print("=" * 80)
print(f"ANNUAL RETURNS · LOT-ADJUSTED · HARD LIMIT -{HARD_LIMIT:.0%} · * = partial year")
print("=" * 80)
display(annual.style.format({"ret": "{:+.1%}", "mdd": "{:.1%}"}))
print(f"Steps below the limit: {report_annual['steps_in_breach']} | first breach: {report_annual['first_breach']}")

print("\nCHECKS (5.2):")
full_y = annual[~annual["partial"]]
neg = full_y[full_y["ret"] <= 0]
n_pos = len(full_y) - len(neg)
breach_years = annual.loc[annual["breach"], "year"].astype(int).tolist()
check("5.2 Annual", "No calendar year breaches the hard limit", not annual["breach"].any(),
      f"years in breach: {breach_years if breach_years else 'none'} | worst year DD {annual['mdd'].min() * 100:.1f}%")
check("5.2 Annual", f">= {YEARS_POS_MIN:.0%} of full calendar years are profitable", n_pos / len(full_y) >= YEARS_POS_MIN,
      f"{n_pos}/{len(full_y)} full years positive" + (" | non-positive: " + ", ".join(f"{int(y)} ({r:+.1%})" for y, r in zip(neg["year"], neg["ret"])) if len(neg) else ""))
plot_annual_returns(annual, report_annual, HARD_LIMIT)

## 6. SANITY CHECKS & VERIFICATION

In [ ]:
print("=" * 70)
print("SANITY CHECKS & REPRODUCIBILITY")
print("=" * 70)

# 1. No lookahead + execution model (violations are collected, not asserted, so the notebook keeps running)
violations, prev_end = [], None
t_in_h, t_in_m = (int(x) for x in ENTRY_JST.split(":"))
t_out_h, t_out_m = (int(x) for x in EXIT_JST.split(":"))
exp_hold = (pd.Timedelta(hours=t_out_h, minutes=t_out_m) - pd.Timedelta(hours=t_in_h, minutes=t_in_m))
for idx_, r in trades_all.iterrows():
    et, xt = pd.Timestamp(r["entry_time"]), pd.Timestamp(r["exit_time"])
    jst_in = et + pd.Timedelta(hours=9)
    if not (df.index[int(r["i_ent"])] == et and df.index[int(r["i_sal"])] == xt):
        violations.append(f"trade {idx_}: i_ent / i_sal misaligned")
    if (jst_in.hour, jst_in.minute) != (t_in_h, t_in_m):
        violations.append(f"trade {idx_}: entry is not {ENTRY_JST} JST")
    if xt - et != exp_hold:
        violations.append(f"trade {idx_}: holding is not {exp_hold}")
    if jst_in.dayofweek >= 5:
        violations.append(f"trade {idx_}: entry on a weekend (JST)")
    if prev_end is not None and et < prev_end:
        violations.append(f"trade {idx_}: overlaps the previous trade")
    prev_end = xt
print(f"Sanity Check 1: entry {ENTRY_JST} JST, exit {EXIT_JST} JST, hold {exp_hold}, no overlap ({len(trades_all)} trades).")
print("   - No indicator: the signal is the calendar, known in advance, so there is no lookahead by construction.")
print("   - Execution at the open of the entry and exit bars, costs per cost_mode.")
check("6. Sanity", "No lookahead and execution model respected on every trade", not violations,
      f"{len(trades_all)} trades checked" + (f" | {len(violations)} violations, first: {violations[0]}" if violations else ""))

# 2. Engine dates vs the calendar rebuilt in this notebook
eng_dates = pd.DatetimeIndex(sorted({(pd.Timestamp(t) + pd.Timedelta(hours=9)).normalize() for t in trades_all["entry_time"]}))
expected = pd.DatetimeIndex(g_win.index)
only_eng, only_nb = eng_dates.difference(expected), expected.difference(eng_dates)
print(f"\nSanity Check 2: engine trade dates vs notebook calendar: {len(eng_dates)} vs {len(expected)} "
      f"| only engine: {len(only_eng)} | only notebook: {len(only_nb)}")
if len(only_eng): print("   only engine (first 10):", [f"{d:%Y-%m-%d}" for d in only_eng[:10]])
if len(only_nb):  print("   only notebook (first 10):", [f"{d:%Y-%m-%d}" for d in only_nb[:10]])
check("6. Sanity", "Engine calendar matches the notebook calendar (>= 99% of dates)", len(only_eng) + len(only_nb) <= 0.01 * len(expected),
      f"{len(only_eng)} only in engine, {len(only_nb)} only in notebook of {len(expected)}")

# 3. Trade distribution per calendar year (full years only)
trades_all["year"] = pd.to_datetime(trades_all["entry_time"]).dt.year
tpy = trades_all.groupby("year").size()
full_years = [y for y in range(df.index[0].year, df.index[-1].year + 1)
              if pd.Timestamp(f"{y}-01-01") >= df.index[0] and pd.Timestamp(f"{y}-12-31") <= df.index[-1]]
tpy_full = tpy.reindex(full_years, fill_value=0)
print("\nSanity Check 3: Trade Frequency per Year:")
print(f"   - Average trades per year: {tpy.mean():.1f}")
print(f"   - Min trades (full years): {tpy_full.min()} ({tpy_full.idxmin()}) | Max trades: {tpy.max()} ({tpy.idxmax()})")
check("6. Sanity", f"Every full calendar year has >= {MIN_TRADES_PER_YEAR} trades", (tpy_full >= MIN_TRADES_PER_YEAR).all(),
      f"min {tpy_full.min()} ({tpy_full.idxmin()}) | max {tpy_full.max()} ({tpy_full.idxmax()}) over {len(tpy_full)} full years")

# 4. MTM paths reproduce the trade returns exactly
paths = eng.trade_mtm_paths(df, trades_all, sizing="capital")
err = max(abs(np.prod(1 + p) - (1 + pnl / 100)) for p, pnl in zip(paths, trades_all["pnl"]))
print(f"\nSanity Check 4: MTM paths telescope to trade returns (max abs error {err:.2e}).")
check("6. Sanity", "MTM paths telescope to the trade returns (error < 1e-9)", err < 1e-9, f"max abs error {err:.2e}")

# 5. The notebook's own gross window return vs the engine's gross return on the same dates
eng_gross = pd.Series(trades_all["gross_pnl"].to_numpy() if "gross_pnl" in trades_all else np.nan,
                      index=(pd.to_datetime(trades_all["entry_time"]) + pd.Timedelta(hours=9)).dt.normalize())
common = eng_gross.index.intersection(g_win.index)
if eng_gross.notna().any() and len(common) > 0:
    diff = (eng_gross.loc[common] - g_win.loc[common, "ret"]).abs().max()
    print(f"\nSanity Check 5: max |engine gross - notebook gross| = {diff:.2e} % over {len(common)} trades.")
    check("6. Sanity", "Engine gross return equals the notebook window return (error < 1e-6 %)", diff < 1e-6, f"max abs diff {diff:.2e} %")
else:
    check("6. Sanity", "Engine gross return equals the notebook window return", None, "engine does not expose gross_pnl")

## 7. REFERENCE VARIANTS: GOTOBI VS NAKED WINDOW VS COST SCENARIOS (SAME BARS, SAME SIZE)

In [ ]:
v_gotobi = trades_all
v_naked = eng.run_gotobi_backtest(df, **{**RULES, "gotobi_days": "all"})            # every business day, same window
cost_runs = {f"Gotobi, flat {fp:g} pips": eng.run_gotobi_backtest(df, **{**RULES, "cost_mode": "flat_pips", "flat_pips": fp}) for fp in FLAT_PIPS}
variants = {"Naked window (every business day)": v_naked, f"Gotobi ({COST_MODE} x {SPREAD_SCALE:g})": v_gotobi, **cost_runs}

rows, curves, raw = [], {}, {}
for name, t in variants.items():
    m = compute_metrics(sized(t, "fraction"), 1.0, 0.0)
    mt = compute_metrics(t, 1.0, 0.0)
    r = eng.daily_returns(df, t, sizing="fraction")
    eq = (1 + r).cumprod()
    curves[name] = eq
    raw[name] = {"pf": mt["profit_factor"], "exp": mt["expectancy"], "sharpe": sharpe_ratio(r, periods_per_year=PPY),
                 "mdd": max_drawdown(eq), "final": float(eq.iloc[-1]), "t": tstat(t["pnl"])}
    rows.append({"Variant": name, "Trades": m["num_trades"], "Win Rate": f"{m['win_rate']:.1f}%", "Profit Factor": f"{mt['profit_factor']:.3f}",
                 "Net t": f"{raw[name]['t']:.2f}", "Sharpe": f"{raw[name]['sharpe']:.2f}", "Max Drawdown": f"{raw[name]['mdd'] * 100:.2f}%",
                 "Final Capital": f"{eq.iloc[-1]:.2f}x"})
print("=" * 80)
print(f"VARIANT COMPARISON ({SIZE_FRAC:.0%} capital per trade)")
print("=" * 80)
display(pd.DataFrame(rows))

print("\nCHECKS (7):")
k_naked, k_g = list(variants)[0], list(variants)[1]
g, nk = raw[k_g], raw[k_naked]
check("7. Variants", "Gotobi Profit Factor beats the naked window", g["pf"] > nk["pf"], f"gotobi {g['pf']:.3f} | naked {nk['pf']:.3f}")
check("7. Variants", "Gotobi expectancy per trade beats the naked window", g["exp"] > nk["exp"], f"gotobi {g['exp']:+.4f}% | naked {nk['exp']:+.4f}%")
check("7. Variants", "Gotobi Sharpe beats the naked window", g["sharpe"] > nk["sharpe"], f"gotobi {g['sharpe']:.2f} | naked {nk['sharpe']:.2f}")
for fp in FLAT_PIPS:
    k = f"Gotobi, flat {fp:g} pips"
    check("7. Variants", f"Survives a flat {fp:g} pips cost (PF >= {PF_MIN:g}, t >= {T_MIN:g})", raw[k]["pf"] >= PF_MIN and raw[k]["t"] >= T_MIN,
          f"PF {raw[k]['pf']:.3f} | t {raw[k]['t']:.2f}")

fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
dark_ax(ax)
for (name, eq), col in zip(curves.items(), (SUB, GREEN, "#5468ff", ORANGE)):
    ax.plot(eq.index, eq, color=col, lw=1.6, label=name)
ax.axvline(oos_start, color=ORANGE, ls="--", lw=1.2, label="OOS start")
ax.set_title("Equity Curve Comparison: naked window vs Gotobi vs cost scenarios", color=TXT, fontweight="bold", loc="left")
ax.legend(frameon=False, labelcolor=SUB, fontsize=9, loc="upper left")
plt.tight_layout(); plt.show()

## 8. SUMMARY OF HYPOTHESIS & NEXT STEPS

In [ ]:
m_all, m_is, m_oos = (compute_metrics(t, 1.0, 0.0) for t in (trades_all, trades_is, trades_oos))
t_all = tstat(trades_all["pnl"])
ret_fr_all = eng.daily_returns(df, trades_all, sizing="fraction")
max_dd_fr = max_drawdown((1 + ret_fr_all).cumprod()) * 100
years = (df.index[-1] - df.index[0]).days / 365.25
next_nb = "02_Gotobi_Optimization.ipynb"

confirmed = bool(m_all["profit_factor"] >= PF_MIN and m_oos["profit_factor"] >= PF_MIN and t_all >= T_MIN and m_all["num_trades"] >= N_MIN)
verdict = "Hypothesis Confirmed" if confirmed else "Hypothesis NOT clearly confirmed"
check("8. Verdict", f"Core criteria: PF >= {PF_MIN:g} (full and OOS), net t >= {T_MIN:g}, N >= {N_MIN}", confirmed,
      f"PF {m_all['profit_factor']:.3f} / OOS {m_oos['profit_factor']:.3f} | t {t_all:.2f} | N {m_all['num_trades']}", quiet=True)

# ---- GO / STOP gate: only the checks that decide whether the research continues ----
# (section, start of the check name, what to do if it FAILS). Remove a line to stop treating it as a gate.
ESSENTIAL = [
    ("6. Sanity",       "No lookahead",                        "Fix the execution model first: every other number is invalid until this passes."),
    ("6. Sanity",       "Engine calendar matches",             "The engine trades different dates than the gotobi calendar: review shift, end-of-month and the JST conversion."),
    ("5. Baseline",     "Profit Factor, IS",                   "Baseline does not hold in-sample: the hypothesis is not supported, rethink the rules."),
    ("5. Baseline",     "Profit Factor, OOS",                  "Baseline does not hold out-of-sample: the edge is not confirmed."),
    ("5. Baseline",     "Net t-stat of the mean trade",        "The mean trade does not clear the multiple-testing bar: the edge could be noise."),
    ("5. Baseline",     "Trades, full period",                 "Too few trades to conclude anything."),
    ("5. Baseline",     "Trades, OOS",                         "Too few OOS trades to validate the edge."),
    ("4.2 Placebos",    "No placebo window is significant",    "A neighbouring window is also significant: the effect is a Tokyo-morning drift, not a gotobi effect."),
    ("4.2 Placebos",    "Real window has the highest mean",    "A placebo window beats the real one: the calendar rule adds nothing."),
    ("4.3 Payment dates", "No single payment date",           "One payment date carries the edge: restrict or rethink the calendar."),
    ("7. Variants",     "Gotobi Profit Factor beats",          "Gotobi does not beat the naked window: the calendar filter is not justified."),
    ("7. Variants",     "Survives a flat 1 pips cost",         "The edge disappears with a 1 pip cost: measure the real Axi spread at 22:30 UTC before going on."),
    ("5.1 Hard limit",  "Lot-adjusted sizing stays inside",    "Lot-adjusted sizing breaks the hard limit: reduce the size fraction."),
    ("5.1 Hard limit",  "Capital >= minimum capital",          "Capital is below what the minimum lot needs."),
    ("5.2 Annual",      "No calendar year breaches",           "A calendar year breaches the hard limit: review sizing or the rules of that year."),
]
chk = pd.DataFrame(CHECKS)
n_pass, n_fail, n_skip = (int((chk["status"] == s).sum()) for s in ("PASS", "FAIL", "SKIP"))
key = {(s, n): a for s, n, a in ESSENTIAL}
chk["essential"] = [any(sec == s and nm.startswith(n) for s, n, _ in ESSENTIAL) for sec, nm in zip(chk["section"], chk["name"])]
ess = chk[chk["essential"]]
missing = [f"{s}: {n}" for s, n, _ in ESSENTIAL if not ((chk["section"] == s) & chk["name"].str.startswith(n)).any()]
ess_fail = ess[ess["status"] == "FAIL"]
go = ess_fail.empty and not missing

print("=" * 80)
print("8. SUMMARY OF HYPOTHESIS & NEXT STEPS")
print("=" * 80)
print(f"\nDECISION: {'GO - continue the research' if go else 'STOP - fix the failed gates before continuing'}   "
      f"({int((ess['status'] == 'PASS').sum())} PASS / {len(ess_fail)} FAIL / {int((ess['status'] == 'SKIP').sum())} SKIP of {len(ESSENTIAL)} essential checks)   |   {verdict}")
if missing:
    print("   Not run (run the sections above first): " + "; ".join(missing))
print("\n" + "-" * 80 + "\nESSENTIAL CHECKS\n" + "-" * 80)
for sec, gr in ess.groupby("section", sort=False):
    print(f"\n{sec}")
    for _, row in gr.iterrows():
        print(f"   [{row['status']}] {row['name']}" + (f"  ->  {row['detail']}" if row["detail"] else ""))

print("\n" + "-" * 80 + "\nKEY FINDINGS\n" + "-" * 80)
print(f"""
1. {verdict}: {m_all['num_trades']} trades over {years:.0f}+ years, win rate {m_all['win_rate']:.1f}%, Profit Factor {m_all['profit_factor']:.3f}
   (IS {m_is['profit_factor']:.3f} / OOS {m_oos['profit_factor']:.3f}), net t-stat of the mean trade {t_all:.2f} (bar: {T_MIN:g}).

2. Specificity: real window mean {plc_tbl.loc[0, 'Gross mean %']:+.4f}% vs best placebo {others['Gross mean %'].max():+.4f}% (max placebo t = {others['Gross t'].max():.2f}).

3. Calendar: top payment date carries {top_share:.0%} of the net edge.

4. Risk at {SIZE_FRAC:.0%} sizing: max daily-MTM drawdown {max_dd_fr:.2f}% over {df.index[0].year}-{df.index[-1].year}; hard limit -{HARD_LIMIT:.0%}.

5. Costs are a placeholder ({COST_MODE} x {SPREAD_SCALE:g}): the real Axi spread at 22:30 UTC is still to be measured with the logger.
""")
print("-" * 80 + "\nNEXT STEPS\n" + "-" * 80)
steps = [key[(r["section"], next(n for s, n, _ in ESSENTIAL if s == r["section"] and r["name"].startswith(n)))] for _, r in ess_fail.iterrows()]
if missing:
    steps.append("Run all the sections above so every essential check is evaluated.")
if go:
    steps.append(f"Proceed to {next_nb} to check that the entry / exit window sits on a broad, stable plateau on in-sample data.")
steps.append("Measure the real Axi spread at 22:30 UTC and replace spread_scale in the variant YAML.")
steps.append("Compute the correlation with the Monday Effect (trades_all of both engines) before sizing the portfolio.")
for i, s in enumerate(steps, 1):
    print(f"{i}. {s}")

## 9. SAVE THE RESULTS OF THIS NOTEBOOK FOR THE VARIANT

In [ ]:
limit_report = eng.hard_limit_report(bar_lot, HARD_LIMIT)

save_json(cfg, "01_hypothesis.json", {
    "variant": cfg["id"],
    "ticker": TICKER,
    "params": cfg["params"],
    "costs": cfg["costs"],
    "oos_start": str(oos_start.date()),
    "all": {"n": int(len(trades_all)), "pf": float(m_all["profit_factor"]), "win_rate": float(m_all["win_rate"]), "t_stat": float(t_all)},
    "is":  {"n": int(len(trades_is)),  "pf": float(m_is["profit_factor"])},
    "oos": {"n": int(len(trades_oos)), "pf": float(m_oos["profit_factor"])},
    "max_dd_fraction_sizing_pct": float(max_dd_fr),
    "hard_limit": {"limit": float(HARD_LIMIT), "max_dd_bar_pct": float(limit_report["max_dd"] * 100), "passed": bool(limit_report["passed"])},
    "window_by_period": win_tbl.round(4).reset_index().to_dict(orient="records"),
    "placebos": plc_tbl.round(4).reset_index().to_dict(orient="records"),
    "payment_dates": dom.round(4).reset_index().to_dict(orient="records"),
    "hypotheses_tested": int(EV["hypotheses_tested"]),
    "min_t_net": float(T_MIN),
    "hypothesis_confirmed": bool(confirmed),
    "checks_summary": {"pass": n_pass, "fail": n_fail, "skip": n_skip},
    "checks": CHECKS,
})